In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
import pandas as pd

# Load dataset
data = pd.read_excel('hospital-dataset.xlsx')

# Remove columns with too many unique values / identifiers
data = data.drop([
    'Name',
    'Doctor',
    'Hospital'
], axis=1)

# Convert dates to datetime
data['Date of Admission'] = pd.to_datetime(data['Date of Admission'])
data['Discharge Date'] = pd.to_datetime(data['Discharge Date'])

# Create useful numerical date features
data['Admission_Year'] = data['Date of Admission'].dt.year
data['Admission_Month'] = data['Date of Admission'].dt.month
data['Admission_Day'] = data['Date of Admission'].dt.day

data['Discharge_Year'] = data['Discharge Date'].dt.year
data['Discharge_Month'] = data['Discharge Date'].dt.month
data['Discharge_Day'] = data['Discharge Date'].dt.day

# Remove original date columns
data = data.drop([
    'Date of Admission',
    'Discharge Date'
], axis=1)

# Convert categorical columns
categorical_columns = data.select_dtypes(include=['object']).columns

data_encoded = pd.get_dummies(
    data,
    columns=categorical_columns,
    drop_first=True,
    dtype='int8'
)

print(data_encoded.head())

print("\nShape:")
print(data_encoded.shape)

   Age  Billing Amount  Room Number  Admission_Year  Admission_Month  \
0   30    18856.281306          328            2024                1   
1   62    33643.327287          265            2019                8   
2   76    27955.096079          205            2022                9   
3   28    37909.782410          450            2020               11   
4   43    14238.317814          458            2022                9   

   Admission_Day  Discharge_Year  Discharge_Month  Discharge_Day  Gender_Male  \
0             31            2024                2              2            1   
1             20            2019                8             26            1   
2             22            2022               10              7            0   
3             18            2020               12             18            0   
4             19            2022               10              9            0   

   ...  Insurance Provider_Medicare  Insurance Provider_UnitedHealthcare  \
0  .

In [16]:
data_encoded
from sklearn.preprocessing import StandardScaler

# Select numerical columns that need scaling
numeric_columns = [
    'Age',
    'Billing Amount',
    'Room Number',
    'Admission_Year',
    'Admission_Month',
    'Admission_Day',
    'Discharge_Year',
    'Discharge_Month',
    'Discharge_Day'
]

# Create scaler
scaler = StandardScaler()

# Scale numerical columns
data_encoded[numeric_columns] = scaler.fit_transform(
    data_encoded[numeric_columns]
)

# Check result
print(data_encoded.head())

        Age  Billing Amount  Room Number  Admission_Year  Admission_Month  \
0 -1.098824       -0.470261     0.233120        1.780122        -1.608576   
1  0.533639        0.570250    -0.313556       -1.559231         0.428167   
2  1.247842        0.169990    -0.834199        0.444381         0.719130   
3 -1.200853        0.870465     1.291761       -0.891360         1.301057   
4 -0.435636       -0.795211     1.361180        0.444381         0.719130   

   Admission_Day  Discharge_Year  Discharge_Month  Discharge_Day  Gender_Male  \
0       1.736213        1.747420        -1.318210      -1.572609            1   
1       0.489660       -1.582907         0.427042       1.166254            1   
2       0.716306        0.415289         1.008792      -1.002012            0   
3       0.263014       -0.916842         1.590543       0.253300            0   
4       0.376337        0.415289         1.008792      -0.773774            0   

   ...  Insurance Provider_Medicare  Insurance Pro

In [17]:
data_encoded

,Age,Billing Amount,Room Number,Admission_Year,Admission_Month,Admission_Day,Discharge_Year,Discharge_Month,Discharge_Day,Gender_Male,...,Insurance Provider_Medicare,Insurance Provider_UnitedHealthcare,Admission Type_Emergency,Admission Type_Urgent,Medication_Ibuprofen,Medication_Lipitor,Medication_Paracetamol,Medication_Penicillin,Test Results_Inconclusive,Test Results_Normal
0,-1.098824,-0.470261,0.233120,1.780122,-1.608576,1.736213,1.747420,-1.318210,-1.572609,1,...,0,0,0,1,0,0,1,0,0,1
1,0.533639,0.570250,-0.313556,-1.559231,0.428167,0.489660,-1.582907,0.427042,1.166254,1,...,1,0,1,0,1,0,0,0,1,0
2,1.247842,0.169990,-0.834199,0.444381,0.719130,0.716306,0.415289,1.008792,-1.002012,0,...,0,0,1,0,0,0,0,0,0,1
3,-1.200853,0.870465,1.291761,-0.891360,1.301057,0.263014,-0.916842,1.590543,0.253300,0,...,1,0,0,0,1,0,0,0,0,0
4,-0.435636,-0.795211,1.361180,0.444381,0.719130,0.376337,0.415289,1.008792,-0.773774,0,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,-0.486651,-1.610589,1.005407,-0.891360,0.428167,0.036367,-0.916842,0.717917,-0.089058,0,...,0,0,0,0,0,0,0,1,0,0
55496,0.482625,0.416462,0.128991,-0.891360,-1.608576,0.829629,-0.916842,-1.318210,-1.686728,0,...,0,0,0,0,0,0,0,0,0,1
55497,-0.690708,0.146464,0.397990,-0.891360,0.137204,-0.303602,-0.916842,0.427042,-0.659655,0,...,0,1,0,1,1,0,0,0,0,0
55498,-0.435636,0.486357,0.172378,-1.559231,-0.444723,1.056275,-1.582907,-0.445584,1.736850,1,...,1,0,0,0,1,0,0,0,0,0


In [5]:
import pandas as pd
import threading
import webbrowser

from flask import Flask, request, render_template_string

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 1. LOAD DATASET
# ============================================================

data = pd.read_excel("hospital-dataset.xlsx")

print("Dataset loaded successfully!")
print("Original shape:", data.shape)


# ============================================================
# 2. REMOVE HIGH-CARDINALITY COLUMNS
# ============================================================

data = data.drop(
    ["Name", "Doctor", "Hospital"],
    axis=1,
    errors="ignore"
)


# ============================================================
# 3. CONVERT DATE COLUMNS
# ============================================================

data["Date of Admission"] = pd.to_datetime(
    data["Date of Admission"],
    errors="coerce"
)

data["Discharge Date"] = pd.to_datetime(
    data["Discharge Date"],
    errors="coerce"
)


# ============================================================
# 4. CREATE DATE FEATURES
# ============================================================

data["Admission_Year"] = data["Date of Admission"].dt.year
data["Admission_Month"] = data["Date of Admission"].dt.month
data["Admission_Day"] = data["Date of Admission"].dt.day

data["Discharge_Year"] = data["Discharge Date"].dt.year
data["Discharge_Month"] = data["Discharge Date"].dt.month
data["Discharge_Day"] = data["Discharge Date"].dt.day

data["Length_of_Stay"] = (
    data["Discharge Date"] -
    data["Date of Admission"]
).dt.days


# Remove original dates
data = data.drop(
    ["Date of Admission", "Discharge Date"],
    axis=1
)


# Remove missing rows
data = data.dropna()


# ============================================================
# 5. FEATURES AND TARGET
# ============================================================

X = data.drop(
    "Medical Condition",
    axis=1
)

y = data["Medical Condition"]


# ============================================================
# 6. IDENTIFY COLUMNS
# ============================================================

categorical_columns = X.select_dtypes(
    include=["object"]
).columns

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns


# ============================================================
# 7. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_columns
        )
    ]
)


# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 9. RANDOM FOREST MODEL
# ============================================================

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

print("Training model...")

model.fit(
    X_train,
    y_train
)

print("Model trained successfully!")


# ============================================================
# 11. CREATE FLASK APPLICATION
# ============================================================

app = Flask(__name__)


# ============================================================
# 12. HTML PAGE
# ============================================================

HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Hospital ML Prediction</title>

    <style>

        * {
            box-sizing: border-box;
        }

        :root {
            --background: #07111f;
            --background-secondary: #0b1728;

            --card: #101d2d;
            --card-secondary: #132235;

            --border: #22344a;
            --border-light: #2b4058;

            --primary: #14b8a6;
            --primary-dark: #0f9488;
            --primary-light: #5eead4;

            --blue: #38bdf8;
            --blue-dark: #0284c7;

            --text: #f8fafc;
            --text-secondary: #cbd5e1;
            --text-muted: #8fa3b8;

            --success: #22c55e;
            --success-bg: #0b2a20;
            --success-border: #17633e;

            --shadow: rgba(0, 0, 0, 0.35);
        }


        /* =====================================================
           BODY
           ===================================================== */

        body {
            margin: 0;
            min-height: 100vh;

            font-family:
                Inter,
                "Segoe UI",
                Roboto,
                Arial,
                sans-serif;

            color: var(--text);

            background:
                radial-gradient(
                    circle at 85% 5%,
                    rgba(20, 184, 166, 0.10),
                    transparent 25%
                ),

                radial-gradient(
                    circle at 10% 90%,
                    rgba(56, 189, 248, 0.07),
                    transparent 25%
                ),

                linear-gradient(
                    135deg,
                    #050c16 0%,
                    #07111f 45%,
                    #0a1626 100%
                );
        }


        /* =====================================================
           MAIN CONTAINER
           ===================================================== */

        .container {
            width: 92%;
            max-width: 1180px;

            margin: auto;

            padding:
                38px 0
                60px;
        }


        /* =====================================================
           HEADER
           ===================================================== */

        .header {
            position: relative;

            overflow: hidden;

            padding: 34px 36px;

            margin-bottom: 24px;

            border-radius: 18px;

            border: 1px solid var(--border);

            background:
                linear-gradient(
                    135deg,
                    #102236 0%,
                    #0d1b2d 55%,
                    #0b1727 100%
                );

            box-shadow:
                0 20px 45px var(--shadow);
        }


        .header::after {
            content: "";

            position: absolute;

            width: 220px;
            height: 220px;

            right: -80px;
            top: -100px;

            border-radius: 50%;

            background:
                rgba(20, 184, 166, 0.10);

            filter: blur(5px);
        }


        .header h1 {
            position: relative;
            z-index: 2;

            margin: 0 0 10px;

            font-size: 30px;

            font-weight: 750;

            letter-spacing: -0.7px;

            color: #f8fafc;
        }


        .header p {
            position: relative;
            z-index: 2;

            margin: 0;

            font-size: 15px;

            color: var(--text-muted);
        }


        /* =====================================================
           CARDS
           ===================================================== */

        .card {
            padding: 28px;

            margin-bottom: 24px;

            border-radius: 16px;

            border: 1px solid var(--border);

            background:
                linear-gradient(
                    145deg,
                    rgba(16, 29, 45, 0.98),
                    rgba(13, 26, 42, 0.98)
                );

            box-shadow:
                0 14px 35px var(--shadow);

            transition:
                border-color 0.2s ease,
                transform 0.2s ease;
        }


        .card:hover {
            border-color: #2c465f;
        }


        .card h2 {
            margin: 0 0 24px;

            font-size: 20px;

            font-weight: 700;

            color: #f1f5f9;
        }


        .card h2::after {
            content: "";

            display: block;

            width: 42px;
            height: 3px;

            margin-top: 9px;

            border-radius: 10px;

            background:
                linear-gradient(
                    90deg,
                    var(--primary),
                    var(--blue)
                );
        }


        /* =====================================================
           FORM GRID
           ===================================================== */

        .form-grid {
            display: grid;

            grid-template-columns:
                repeat(3, minmax(0, 1fr));

            gap: 21px;
        }


        /* =====================================================
           LABELS
           ===================================================== */

        label {
            display: block;

            margin-bottom: 8px;

            font-size: 13px;

            font-weight: 650;

            color: var(--text-secondary);

            letter-spacing: 0.1px;
        }


        /* =====================================================
           INPUTS / SELECTS
           ===================================================== */

        input,
        select {
            width: 100%;

            padding:
                12px 14px;

            border-radius: 9px;

            border: 1px solid #2b4057;

            outline: none;

            background: #0a1727;

            color: #f8fafc;

            font-family: inherit;

            font-size: 14px;

            transition:
                border-color 0.2s ease,
                box-shadow 0.2s ease,
                background 0.2s ease;
        }


        input:hover,
        select:hover {
            border-color: #3b526b;

            background: #0c1a2c;
        }


        input:focus,
        select:focus {
            border-color: var(--primary);

            background: #0d1d2d;

            box-shadow:
                0 0 0 3px
                rgba(20, 184, 166, 0.13);
        }


        input::placeholder {
            color: #60748a;
        }


        input[type="date"]::-webkit-calendar-picker-indicator {
            filter: invert(1);

            opacity: 0.65;

            cursor: pointer;
        }


        option {
            background: #0d1b2b;

            color: #f8fafc;
        }


        /* =====================================================
           PREDICT BUTTON
           ===================================================== */

        button {
            width: 100%;

            margin-top: 26px;

            padding: 14px 20px;

            border: none;

            border-radius: 10px;

            color: #ffffff;

            background:
                linear-gradient(
                    135deg,
                    #0d9488 0%,
                    #14b8a6 55%,
                    #2dd4bf 100%
                );

            font-family: inherit;

            font-size: 15px;

            font-weight: 750;

            letter-spacing: 0.15px;

            cursor: pointer;

            box-shadow:
                0 9px 25px
                rgba(20, 184, 166, 0.20);

            transition:
                transform 0.18s ease,
                box-shadow 0.18s ease,
                filter 0.18s ease;
        }


        button:hover {
            transform: translateY(-2px);

            filter: brightness(1.06);

            box-shadow:
                0 13px 30px
                rgba(20, 184, 166, 0.30);
        }


        button:active {
            transform: translateY(0);
        }


        /* =====================================================
           PREDICTION RESULT
           ===================================================== */

        .result {
            position: relative;

            overflow: hidden;

            padding: 21px 23px;

            border-radius: 12px;

            border: 1px solid var(--success-border);

            background:
                linear-gradient(
                    135deg,
                    #09251d,
                    #0b3024
                );

            color: #bbf7d0;

            font-size: 19px;

            font-weight: 650;

            box-shadow:
                0 10px 25px
                rgba(0, 0, 0, 0.20);
        }


        .result::before {
            content: "";

            position: absolute;

            left: 0;
            top: 0;
            bottom: 0;

            width: 4px;

            background: var(--success);
        }


        .result strong {
            color: #4ade80;

            font-weight: 800;
        }


        /* =====================================================
           MODEL INFORMATION
           ===================================================== */

        .info-row {
            display: flex;

            align-items: center;

            justify-content: space-between;

            gap: 20px;

            padding: 15px 0;

            border-bottom:
                1px solid rgba(43, 64, 87, 0.65);
        }


        .info-row:last-child {
            border-bottom: none;

            padding-bottom: 0;
        }


        .info-label {
            color: var(--text-muted);

            font-size: 14px;
        }


        .info-value {
            color: #e2e8f0;

            font-size: 14px;

            font-weight: 650;

            text-align: right;
        }


        .info-row:first-child .info-value {
            color: var(--primary-light);
        }


        /* =====================================================
           RESPONSIVE - TABLET
           ===================================================== */

        @media (max-width: 950px) {

            .form-grid {
                grid-template-columns:
                    repeat(2, minmax(0, 1fr));
            }

        }


        /* =====================================================
           RESPONSIVE - MOBILE
           ===================================================== */

        @media (max-width: 620px) {

            .container {
                width: 94%;

                padding:
                    20px 0
                    35px;
            }


            .header {
                padding: 25px 22px;

                border-radius: 14px;
            }


            .header h1 {
                font-size: 23px;

                line-height: 1.3;
            }


            .header p {
                font-size: 13px;

                line-height: 1.5;
            }


            .card {
                padding: 21px;

                border-radius: 14px;
            }


            .card h2 {
                font-size: 18px;
            }


            .form-grid {
                grid-template-columns: 1fr;

                gap: 16px;
            }


            .info-row {
                align-items: flex-start;

                flex-direction: column;

                gap: 5px;
            }


            .info-value {
                text-align: left;
            }


            .result {
                font-size: 16px;

                line-height: 1.6;
            }

        }

    </style>
</head>


<body>

<div class="container">

    <div class="header">

        <h1>
            🏥 Hospital Medical Condition Prediction
        </h1>

        <p>
            Machine Learning Prediction System
            powered by Random Forest
        </p>

    </div>


    <div class="card">

        <h2>
            Patient Information
        </h2>


        <form method="POST">

            <div class="form-grid">

                <div>

                    <label>
                        Age
                    </label>

                    <input
                        type="number"
                        name="age"
                        value="30"
                        required
                    >

                </div>


                <div>

                    <label>
                        Billing Amount
                    </label>

                    <input
                        type="number"
                        step="0.01"
                        name="billing"
                        value="20000"
                        required
                    >

                </div>


                <div>

                    <label>
                        Room Number
                    </label>

                    <input
                        type="number"
                        name="room"
                        value="100"
                        required
                    >

                </div>


                <div>

                    <label>
                        Gender
                    </label>

                    <select name="gender">

                        {% for value in genders %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Blood Type
                    </label>

                    <select name="blood">

                        {% for value in blood_types %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Insurance Provider
                    </label>

                    <select name="insurance">

                        {% for value in insurances %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Admission Type
                    </label>

                    <select name="admission">

                        {% for value in admissions %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Medication
                    </label>

                    <select name="medication">

                        {% for value in medications %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Test Results
                    </label>

                    <select name="test">

                        {% for value in tests %}

                        <option value="{{ value }}">
                            {{ value }}
                        </option>

                        {% endfor %}

                    </select>

                </div>


                <div>

                    <label>
                        Date of Admission
                    </label>

                    <input
                        type="date"
                        name="admission_date"
                        value="2024-01-01"
                        required
                    >

                </div>


                <div>

                    <label>
                        Discharge Date
                    </label>

                    <input
                        type="date"
                        name="discharge_date"
                        value="2024-01-05"
                        required
                    >

                </div>


            </div>


            <button type="submit">

                🔍 Predict Medical Condition

            </button>


        </form>

    </div>


    {% if prediction %}

    <div class="card">

        <h2>
            Prediction Result
        </h2>


        <div class="result">

            Predicted Medical Condition:

            <strong>
                {{ prediction }}
            </strong>

        </div>

    </div>

    {% endif %}


    <div class="card">

        <h2>
            Model Information
        </h2>


        <div class="info-row">

            <span class="info-label">
                Algorithm
            </span>

            <span class="info-value">
                Random Forest Classifier
            </span>

        </div>


        <div class="info-row">

            <span class="info-label">
                Total Records
            </span>

            <span class="info-value">
                {{ total_records }}
            </span>

        </div>


        <div class="info-row">

            <span class="info-label">
                Training Records
            </span>

            <span class="info-value">
                {{ training_records }}
            </span>

        </div>


        <div class="info-row">

            <span class="info-label">
                Testing Records
            </span>

            <span class="info-value">
                {{ testing_records }}
            </span>

        </div>

    </div>

</div>

</body>
</html>
"""


# ============================================================
# 13. FLASK ROUTE
# ============================================================

@app.route("/", methods=["GET", "POST"])
def home():

    prediction = None

    if request.method == "POST":

        # Patient values

        age = int(
            request.form["age"]
        )

        billing = float(
            request.form["billing"]
        )

        room = int(
            request.form["room"]
        )

        gender = request.form["gender"]

        blood = request.form["blood"]

        insurance = request.form["insurance"]

        admission = request.form["admission"]

        medication = request.form["medication"]

        test = request.form["test"]


        # Dates

        admission_date = pd.to_datetime(
            request.form["admission_date"]
        )

        discharge_date = pd.to_datetime(
            request.form["discharge_date"]
        )


        # Date features

        admission_year = admission_date.year
        admission_month = admission_date.month
        admission_day = admission_date.day

        discharge_year = discharge_date.year
        discharge_month = discharge_date.month
        discharge_day = discharge_date.day


        # Length of stay

        length_of_stay = (
            discharge_date -
            admission_date
        ).days


        # Create patient dataframe

        patient = pd.DataFrame({

            "Age": [age],

            "Billing Amount": [billing],

            "Room Number": [room],

            "Gender": [gender],

            "Blood Type": [blood],

            "Insurance Provider": [insurance],

            "Admission Type": [admission],

            "Medication": [medication],

            "Test Results": [test],

            "Admission_Year": [
                admission_year
            ],

            "Admission_Month": [
                admission_month
            ],

            "Admission_Day": [
                admission_day
            ],

            "Discharge_Year": [
                discharge_year
            ],

            "Discharge_Month": [
                discharge_month
            ],

            "Discharge_Day": [
                discharge_day
            ],

            "Length_of_Stay": [
                length_of_stay
            ]
        })


        # Predict

        prediction = model.predict(
            patient
        )[0]


    return render_template_string(

        HTML,

        prediction=prediction,

        genders=sorted(
            data["Gender"].unique()
        ),

        blood_types=sorted(
            data["Blood Type"].unique()
        ),

        insurances=sorted(
            data["Insurance Provider"].unique()
        ),

        admissions=sorted(
            data["Admission Type"].unique()
        ),

        medications=sorted(
            data["Medication"].unique()
        ),

        tests=sorted(
            data["Test Results"].unique()
        ),

        total_records=len(data),

        training_records=len(X_train),

        testing_records=len(X_test)
    )


# ============================================================
# 14. START SERVER
# ============================================================

def run_app():

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run_app
)

thread.daemon = True

thread.start()


print()
print("==========================================")
print("🚀 WEB APP STARTED")
print("==========================================")
print()
print("Open this URL in your browser:")
print("http://127.0.0.1:5000")
print()
print("You can also use:")
print("http://localhost:5000")
print()

Dataset loaded successfully!
Original shape: (55500, 15)
Training model...
Model trained successfully!

🚀 WEB APP STARTED

Open this URL in your browser:
http://127.0.0.1:5000

You can also use:
http://localhost:5000

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
